[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/05_scalability_apps/05_scalability_apps.ipynb)

# 05 · 可扩展与应用（用 numpy 从零）

目标：从零实现**链接预测**(编码器+点积解码+负采样+AUC) 与**图级分类**(置换不变 readout)，并诊断过平滑的缓解。

路线：负采样 → AUC(秩统计量) → 链接预测训练(删测试边, 验证 AUC 上升) → readout(sum/mean/max + 置换不变) → 图级分类训练 → 残差缓解过平滑 → ✏️ 练习 → 📖 答案 → 🧪 多图分类胶囊。

> 心智模型：**编码器(前四模块)通用，任务靠头——节点级=逐点softmax、边级=成对解码+负采样(看AUC)、图级=readout(置换不变)+MLP。**

## 1 · 负采样：图很稀疏，非边才是大多数

链接预测把「有没有边」当二分类。正样本=真实边($O(|E|)$ 个)，负样本=不存在的节点对($O(n^2)$ 个，绝大多数)。
**负采样**：每个正样本配采样的非边，平衡正负。我们从零实现并验证采样的确实是非边。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def sample_negative_edges(A, num, seed=0):
    '''随机采样 num 个「不存在的边」(i<j 且 A[i,j]=0, i≠j)。'''
    rng = np.random.default_rng(seed)
    nn = len(A)
    negs = []
    while len(negs) < num:
        i, j = rng.integers(0, nn, 2)
        if i != j and A[i, j] == 0:
            negs.append((min(i,j), max(i,j)))
    return negs

def positive_edges(A):
    '''取所有真实边 (i<j)。'''
    return [(i, j) for i in range(len(A)) for j in range(i+1, len(A)) if A[i,j] > 0]

edges = [(0,1),(1,2),(0,2),(3,4),(4,5),(3,5),(2,3)]
n = 6; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
pos = positive_edges(A)
neg = sample_negative_edges(A, num=len(pos), seed=0)
print(f'正样本(真实边) {len(pos)} 个:', pos)
print(f'负样本(采样非边) {len(neg)} 个:', neg)
# 验证：负样本都不是真实边
for i, j in neg:
    assert A[i, j] == 0, f'负样本 ({i},{j}) 不应是真实边'
assert len(neg) == len(pos), '正负样本数量应平衡'
print('✅ 负采样正确：采的全是非边，正负数量平衡')

## 2 · AUC：随机一对正负样本，正样本分更高的概率

链接预测正负极不平衡，**准确率会骗人**(全预测无边也很高)。用 **AUC**：
`AUC = P(score(正) > score(负))`(平局算半个)。我们用秩统计量从零实现并对拍。

In [ ]:
def auc_score(pos_scores, neg_scores):
    '''Mann-Whitney: 统计所有(正,负)对中正样本得分更高的比例(平局0.5)。'''
    pos = np.asarray(pos_scores); neg = np.asarray(neg_scores)
    wins = 0.0
    for ps in pos:
        wins += np.sum(ps > neg) + 0.5 * np.sum(ps == neg)
    return wins / (len(pos) * len(neg))

# 完美排序：所有正样本分都高于所有负样本 -> AUC=1
assert auc_score([3,4,5], [0,1,2]) == 1.0
# 完全反 -> AUC=0
assert auc_score([0,1,2], [3,4,5]) == 0.0
# 随机/无区分 -> AUC≈0.5
assert auc_score([1,2,3], [1,2,3]) == 0.5
# 部分重叠手算: pos=[2,4] neg=[1,3] -> (2>1)+(2>3=0)+(4>1)+(4>3)=3/4=0.75
assert auc_score([2,4],[1,3]) == 0.75
print('✅ AUC 从零实现正确：完美=1, 反序=0, 随机=0.5, 手算对拍 0.75')

## 3 · 链接预测训练：删测试边 → 训练 → AUC 上升

完整流程：① **从图删掉测试边**(防泄漏！)；② GCN 编码器得节点嵌入；③ 点积解码 `σ(z_u·z_v)`；
④ 正样本+负采样二元交叉熵训练；⑤ 在测试边上算 AUC。**正确性=测试 AUC 显著高于 0.5**。

In [ ]:
def gcn_norm(A):
    At = A + np.eye(len(A)); d = At.sum(1)
    di = np.diag(1/np.sqrt(d)); return di @ At @ di
def relu(x): return np.maximum(x, 0.0)
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-30,30)))

def make_sbm(sizes, p_in, p_out, seed=0):
    rng = np.random.default_rng(seed); nn=sum(sizes)
    lab = np.concatenate([np.full(s,k) for k,s in enumerate(sizes)])
    A=np.zeros((nn,nn))
    for i in range(nn):
        for j in range(i+1,nn):
            if rng.random()<(p_in if lab[i]==lab[j] else p_out): A[i,j]=A[j,i]=1.0
    return A, lab

# 造图，划分训练/测试边
A_full, _ = make_sbm([20,20], 0.3, 0.05, seed=1)
nn = len(A_full)
all_pos = positive_edges(A_full)
rng = np.random.default_rng(2)
rng.shuffle(all_pos)
n_test = len(all_pos)//5
test_pos = all_pos[:n_test]; train_pos = all_pos[n_test:]
# 关键：训练图删掉测试边(防泄漏)
A_train = A_full.copy()
for i,j in test_pos: A_train[i,j]=A_train[j,i]=0.0
print(f'训练边 {len(train_pos)}, 测试边 {len(test_pos)}（已从训练图删除）')

In [ ]:
# 简单编码器：2 层 GCN(无监督，靠链接预测损失驱动). 特征用单位阵
rng = np.random.default_rng(3)
Ah = gcn_norm(A_train)
X = np.eye(nn)
d_emb = 16
W0 = rng.standard_normal((nn, 32))*0.3
W1 = rng.standard_normal((32, d_emb))*0.3
def encode(W0, W1):
    H1 = relu(Ah @ X @ W0)
    return Ah @ H1 @ W1            # 节点嵌入 z

def edge_scores(Z, edge_list):
    idx = np.array(edge_list)
    return (Z[idx[:,0]] * Z[idx[:,1]]).sum(1)   # 点积解码

def encode_pre(W0, W1):
    pre = Ah @ X @ W0; H1 = relu(pre)
    return Ah @ H1 @ W1, pre        # 返回嵌入 Z 与中间量(供反传)

lr = 0.2; aucs=[]; losses=[]
for epoch in range(300):
    Z, pre = encode_pre(W0, W1)
    neg = sample_negative_edges(A_train, num=len(train_pos), seed=epoch)
    ps = edge_scores(Z, train_pos); ns = edge_scores(Z, neg)
    pp = sigmoid(ps); pn = sigmoid(ns)
    loss = -(np.log(pp+1e-9).mean() + np.log(1-pn+1e-9).mean())  # 二元交叉熵
    losses.append(loss)
    # 解析梯度: d loss/d score_pos = -(1-pp)/Npos ; d loss/d score_neg = pn/Nneg
    gZ = np.zeros_like(Z)
    gp = -(1-pp)/len(train_pos); gn = pn/len(neg)
    for k,(u,v) in enumerate(train_pos):
        gZ[u]+=gp[k]*Z[v]; gZ[v]+=gp[k]*Z[u]
    for k,(u,v) in enumerate(neg):
        gZ[u]+=gn[k]*Z[v]; gZ[v]+=gn[k]*Z[u]
    # 反传到 W1, W0
    H1 = relu(pre)
    dW1 = (Ah @ H1).T @ gZ
    dH1 = (Ah @ gZ) @ W1.T
    dpre = dH1 * (pre > 0)
    dW0 = (Ah @ X).T @ dpre
    W0 -= lr*dW0; W1 -= lr*dW1
    if epoch == 0 or epoch == 299:
        Zt, _ = encode_pre(W0,W1)
        test_neg = sample_negative_edges(A_full, num=len(test_pos), seed=999)
        aucs.append(auc_score(edge_scores(Zt,test_pos), edge_scores(Zt,test_neg)))
print(f'测试 AUC: 初={aucs[0]:.3f} -> 末={aucs[-1]:.3f}')
assert aucs[-1] > aucs[0], 'AUC 应随训练上升'
assert aucs[-1] > 0.75, '训好的链接预测器测试 AUC 应明显高于 0.5'
print('✅ 链接预测训练成功：测试 AUC 显著高于随机(0.5)且随训练上升')

## 4 · readout：把节点表示汇成图向量（必须置换不变）

图级任务要一个图向量。readout `h_G = AGG_v(h_v)` 必须**置换不变**(换节点编号图标签不变)。
实现 sum/mean/max 三种，验证置换不变性，并展示 sum 表达力最强(能区分 mean/max 分不开的图)。

In [ ]:
def readout(H, mode='sum'):
    if mode=='sum':  return H.sum(0)
    if mode=='mean': return H.mean(0)
    if mode=='max':  return H.max(0)

H = np.random.default_rng(0).standard_normal((6, 4))
perm = np.random.default_rng(1).permutation(6)
for mode in ['sum','mean','max']:
    assert np.allclose(readout(H, mode), readout(H[perm], mode)), f'{mode} 必须置换不变'
print('✅ sum/mean/max readout 全部置换不变（换节点编号，图向量不变）')

In [ ]:
# sum 表达力 > mean/max：区分「2个相同节点」vs「4个相同节点」
h = np.array([1.0, 2.0])
G2 = np.tile(h, (2,1))      # 图A：2 个节点
G4 = np.tile(h, (4,1))      # 图B：4 个节点(同样的节点特征)
# mean 分不开(都=h)，max 也分不开(都=h)，只有 sum 不同
assert np.allclose(readout(G2,'mean'), readout(G4,'mean')), 'mean 分不开'
assert np.allclose(readout(G2,'max'), readout(G4,'max')), 'max 分不开'
assert not np.allclose(readout(G2,'sum'), readout(G4,'sum')), 'sum 能区分(含数量信息)'
print('sum(G2)=', readout(G2,'sum'), ' sum(G4)=', readout(G4,'sum'))
print('✅ sum 表达力最强：能区分节点数量(多重集)，mean/max 不能 —— GIN 的核心洞察')

## 5 · 图级分类：编码器 + readout + MLP（多图训练）

造两类小图(稠密 vs 稀疏)，用 GCN 编码 + sum readout + 线性分类训练。**正确性=训练损失下降+分类准确**。

> **关键教训**：节点特征用**度**(degree)而非全1。因为对称归一化 GCN 几乎保持常信号，全1特征会让稀疏/稠密图的 readout 几乎一样——**度是最基本的结构特征**，没有它 GCN 看不出稀疏稠密的区别。

In [ ]:
def make_graph(kind, n_nodes=8, seed=0):
    '''kind=0: 稀疏(环); kind=1: 稠密(近全连接)。返回 (A, X), X 用度特征。'''
    rng = np.random.default_rng(seed)
    A = np.zeros((n_nodes, n_nodes))
    if kind == 0:
        for i in range(n_nodes): A[i,(i+1)%n_nodes]=A[(i+1)%n_nodes,i]=1.0  # 环
    else:
        for i in range(n_nodes):
            for j in range(i+1,n_nodes):
                if rng.random()<0.7: A[i,j]=A[j,i]=1.0                       # 稠密
    deg = A.sum(1, keepdims=True)
    X = np.concatenate([np.ones((n_nodes,1)), deg, deg**2/n_nodes], axis=1)  # 度特征(结构信号)
    return A, X

# 数据集：每类 15 张图
dataset = []
for s in range(15):
    dataset.append((make_graph(0, 8, s), 0))
    dataset.append((make_graph(1, 8, s+100), 1))
rng = np.random.default_rng(0); rng.shuffle(dataset)
split = int(0.7*len(dataset)); train_ds, test_ds = dataset[:split], dataset[split:]
print(f'图分类数据集：{len(dataset)} 张图(2类)，训练 {len(train_ds)}，测试 {len(test_ds)}')

In [ ]:
def softmax(z):
    z=z-z.max(); e=np.exp(z); return e/e.sum()

def graph_embed(A, X, W0):
    '''1 层 GCN + sum readout -> 图向量。'''
    H = relu(gcn_norm(A) @ X @ W0)
    return readout(H, 'sum')

rng = np.random.default_rng(5)
W0 = rng.standard_normal((3, 16))*0.3      # GCN 权重
Wc = rng.standard_normal((16, 2))*0.3      # 分类头
lr = 0.02; losses=[]
for epoch in range(120):
    grad_W0 = np.zeros_like(W0); grad_Wc = np.zeros_like(Wc); tot=0
    for (A,X), y in train_ds:
        H = relu(gcn_norm(A) @ X @ W0)
        hg = H.sum(0)
        logits = hg @ Wc; P = softmax(logits)
        tot += -np.log(P[y]+1e-9)
        dlog = P.copy(); dlog[y]-=1
        grad_Wc += np.outer(hg, dlog)
        dhg = Wc @ dlog
        # 反传过 sum readout 与 GCN(简化)
        An = gcn_norm(A)
        dH = np.tile(dhg, (len(A),1))
        dpre = dH * ((An @ X @ W0)>0)
        grad_W0 += (An @ X).T @ dpre
    W0 -= lr*grad_W0/len(train_ds); Wc -= lr*grad_Wc/len(train_ds)
    losses.append(tot/len(train_ds))
# 评估
correct=0
for (A,X), y in test_ds:
    hg = relu(gcn_norm(A) @ X @ W0).sum(0)
    if (hg @ Wc).argmax()==y: correct+=1
test_acc = correct/len(test_ds)
print(f'图分类: 初损失={losses[0]:.3f} -> 末={losses[-1]:.3f}, 测试准确率={test_acc:.2%}')
assert losses[-1] < losses[0], '损失应下降'
assert test_acc >= 0.8, '稀疏vs稠密图应能高准确区分'
print('✅ 图级分类训练成功：损失下降、测试准确率高(靠结构区分稀疏/稠密图)')

## 6 · 残差连接缓解过平滑

模块02 看到深层 GCN 过平滑(Dirichlet 能量→0)。一个简单解药：**残差连接** `H' = H + GCN(H)`。
对比深层「纯 GCN」与「GCN+残差」的 Dirichlet 能量，验证残差保住了表示的区分度。

In [ ]:
def dirichlet_energy(A, H):
    L = np.diag(A.sum(1)) - A
    return float(np.trace(H.T @ L @ H))

A_sbm, _ = make_sbm([15,15], 0.4, 0.05, seed=3)
Ah = gcn_norm(A_sbm)
rng = np.random.default_rng(7)
H0 = rng.standard_normal((len(A_sbm), 8))

# 纯 GCN 传播(无参数，看平滑)
H_plain = H0.copy()
for _ in range(20): H_plain = Ah @ H_plain
# GCN + 残差
H_res = H0.copy()
for _ in range(20): H_res = H_res + Ah @ H_res
H_res = H_res / np.linalg.norm(H_res)        # 仅防数值爆炸，不影响相对差异
H_plain_n = H_plain / (np.linalg.norm(H_plain)+1e-12)

e_plain = dirichlet_energy(A_sbm, H_plain_n)
e_res = dirichlet_energy(A_sbm, H_res)
print(f'20 层后 Dirichlet 能量: 纯GCN={e_plain:.4e}, GCN+残差={e_res:.4e}')
assert e_res > e_plain, '残差应保住更多表示区分度(更高能量=更不平滑)'
print('✅ 残差连接缓解过平滑：保住了节点表示的区分度')

---
## ✏️ 练习 1：带多个负样本的负采样

实现 `sample_k_negatives(A, pos_edges, k, seed)`：为**每个**正样本采样 `k` 个负样本(非边)，
返回长度 `k*len(pos_edges)` 的负样本列表。验证全是非边、数量正确。

In [ ]:
def sample_k_negatives(A, pos_edges, k, seed=0):
    # TODO: 为每个正样本采 k 个非边(i≠j, A[i,j]==0)，汇总返回
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
pos = positive_edges(A)
negs = sample_k_negatives(A, pos, k=3, seed=1)
assert len(negs) == 3*len(pos), '应为每个正样本采 k 个'
for i,j in negs: assert A[i,j]==0 and i!=j, '必须是非边'
print(f'✅ 练习 1 通过：{len(pos)} 个正样本 × 3 = {len(negs)} 个负样本，全是非边')

## ✏️ 练习 2：mean+max 拼接 readout

实现 `readout_concat(H)`：把 mean 池化和 max 池化**拼接**成一个 2d 维图向量(实践常用，取长补短)。
验证：维度=2d、置换不变、前半=mean 后半=max。

In [ ]:
def readout_concat(H):
    # TODO: 返回 concat([mean_v h_v, max_v h_v])，长度 2*d
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
H = np.random.default_rng(2).standard_normal((7, 5))
hg = readout_concat(H)
assert hg.shape == (10,), '应为 2d 维'
assert np.allclose(hg[:5], H.mean(0)) and np.allclose(hg[5:], H.max(0))
perm = np.random.default_rng(3).permutation(7)
assert np.allclose(hg, readout_concat(H[perm])), '必须置换不变'
print('✅ 练习 2 通过：mean‖max 拼接 readout，2d 维、置换不变')

## ✏️ 练习 3：AUC 的向量化实现

练习 2 节的 AUC 用了循环。实现 `auc_fast(pos_scores, neg_scores)`：用排序/广播**向量化**算 AUC，
结果与 `auc_score` 一致但更快。提示：可用 `pos[:,None] > neg[None,:]` 广播比较。

In [ ]:
def auc_fast(pos_scores, neg_scores):
    # TODO: 用广播 pos[:,None] vs neg[None,:] 算 (>) + 0.5*(==) 的均值
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng = np.random.default_rng(0)
ps = rng.standard_normal(50); ns = rng.standard_normal(60) - 0.5
assert abs(auc_fast(ps, ns) - auc_score(ps, ns)) < 1e-9, '应与循环版一致'
assert auc_fast([3,4,5],[0,1,2]) == 1.0
assert auc_fast([2,4],[1,3]) == 0.75
print('✅ 练习 3 通过：向量化 AUC 与循环版一致')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sample_k_negatives(A, pos_edges, k, seed=0):
    out = []
    for idx in range(len(pos_edges)):
        out += sample_negative_edges(A, k, seed=seed*1000 + idx)
    return out

In [ ]:
# 练习 2 参考答案
def readout_concat(H):
    return np.concatenate([H.mean(0), H.max(0)])

In [ ]:
# 练习 3 参考答案
def auc_fast(pos_scores, neg_scores):
    pos = np.asarray(pos_scores)[:, None]; neg = np.asarray(neg_scores)[None, :]
    return float(((pos > neg) + 0.5*(pos == neg)).mean())

---
## 🧪 真实数据胶囊：分子式图分类（有环 vs 无环）

分子图分类是 GNN 最成功的应用之一(如毒性/溶解度预测)。一个核心结构判别：**分子里有没有环**(芳香环等)往往与性质强相关。
我们造一批「含环」和「树状(无环)」的小图，用 GNN+readout 区分——模拟「分子有无某结构子图」这类真实任务。

In [ ]:
def make_cyclic_or_tree(has_cycle, n_nodes=7, seed=0):
    '''has_cycle=True: 含环; False: 树(无环)。返回 (A, X).'''
    rng = np.random.default_rng(seed)
    A = np.zeros((n_nodes, n_nodes))
    # 先连成树(随机)
    for v in range(1, n_nodes):
        u = rng.integers(0, v); A[u,v]=A[v,u]=1.0
    if has_cycle:
        added = 0
        while added < 3:                     # 加 3 条边 -> 明显含环、更稠密
            i,j = rng.integers(0,n_nodes,2)
            if i!=j and A[i,j]==0: A[i,j]=A[j,i]=1.0; added+=1
    deg = A.sum(1, keepdims=True)
    X = np.concatenate([np.ones((n_nodes,1)), deg, deg**2/n_nodes], axis=1)  # 度特征
    return A, X

def has_cycle_check(A):
    '''边数 >= 节点数 => 含环(连通图).'''
    return A.sum()/2 >= len(A)

mol_ds = []
for s in range(20):
    mol_ds.append((make_cyclic_or_tree(True, 7, s), 1))
    mol_ds.append((make_cyclic_or_tree(False, 7, s+50), 0))
# 验证标签与真实结构一致
for (A,X), y in mol_ds:
    assert has_cycle_check(A) == bool(y), '标签应与「是否含环」一致'
print(f'分子式图数据集：{len(mol_ds)} 张图(含环 vs 树)，标签与真实结构一致 ✅')

**🧪 胶囊练习**：用 GCN(1层) + sum readout + 线性分类，区分含环/无环图。补全图嵌入那一行(GCN+readout)，
训练后报告测试准确率。提示：含环图边更多，sum readout 后的图向量量级不同。

In [ ]:
def train_graph_clf(dataset, epochs=120, lr=0.02, seed=0):
    rng = np.random.default_rng(seed)
    rng.shuffle(dataset)
    split = int(0.7*len(dataset)); tr, te = dataset[:split], dataset[split:]
    W0 = rng.standard_normal((3,16))*0.3; Wc = rng.standard_normal((16,2))*0.3
    for _ in range(epochs):
        gW0=np.zeros_like(W0); gWc=np.zeros_like(Wc)
        for (A,X), y in tr:
            hg = None     # TODO: relu(gcn_norm(A) @ X @ W0).sum(0)  —— GCN + sum readout
            P = softmax(hg @ Wc)
            dlog = P.copy(); dlog[y]-=1
            gWc += np.outer(hg, dlog)
            An = gcn_norm(A); dH = np.tile(Wc @ dlog, (len(A),1))
            gW0 += (An @ X).T @ (dH * ((An@X@W0)>0))
        W0 -= lr*gW0/len(tr); Wc -= lr*gWc/len(tr)
    correct = sum(int((relu(gcn_norm(A)@X@W0).sum(0)@Wc).argmax()==y) for (A,X),y in te)
    return correct/len(te)

raise NotImplementedError  # 删除并补全 hg

In [ ]:
# 自测
acc = train_graph_clf([(g,y) for g,y in mol_ds], seed=1)
print(f'含环 vs 无环 图分类测试准确率 = {acc:.2%}')
assert acc >= 0.8, 'GNN+readout 应能区分含环/无环图'
print('✅ 胶囊练习通过：GNN+readout 学会了判别「图里有没有环」(真实分子任务的缩影)')

In [ ]:
# 📖 胶囊参考答案
def train_graph_clf(dataset, epochs=120, lr=0.02, seed=0):
    rng = np.random.default_rng(seed); rng.shuffle(dataset)
    split = int(0.7*len(dataset)); tr, te = dataset[:split], dataset[split:]
    W0 = rng.standard_normal((3,16))*0.3; Wc = rng.standard_normal((16,2))*0.3
    for _ in range(epochs):
        gW0=np.zeros_like(W0); gWc=np.zeros_like(Wc)
        for (A,X), y in tr:
            hg = relu(gcn_norm(A) @ X @ W0).sum(0)    # GCN + sum readout
            P = softmax(hg @ Wc); dlog = P.copy(); dlog[y]-=1
            gWc += np.outer(hg, dlog)
            An = gcn_norm(A); dH = np.tile(Wc @ dlog, (len(A),1))
            gW0 += (An @ X).T @ (dH * ((An@X@W0)>0))
        W0 -= lr*gW0/len(tr); Wc -= lr*gWc/len(tr)
    correct = sum(int((relu(gcn_norm(A)@X@W0).sum(0)@Wc).argmax()==y) for (A,X),y in te)
    return correct/len(te)

acc = train_graph_clf([(g,y) for g,y in mol_ds], seed=1)
print(f'含环 vs 无环 图分类测试准确率 = {acc:.2%}')
assert acc >= 0.8
print('✅ GNN+readout 学会判别图的全局结构(有无环)——分子性质预测的缩影')

### 小结 & 全课收官
- **可扩展**：采样/子图/分块 = 用局部子图近似整图传播 + mini-batch SGD。
- **过平滑(越深越差, Dirichlet→0) ≠ 过挤压(长程学不会, 谱隙小)**——别搞混，对症下药。
- **链接预测**：编码器+点积解码+负采样，**删测试边防泄漏**，**看 AUC 不看准确率**。
- **图分类**：置换不变 readout(sum 表达力最强)+MLP；编码器通用，任务靠头。

**🎓 六模块走完**：图拉普拉斯 → GCN → GAT/SAGE → 图 Transformer → 链接预测/图分类。
你已从零实现并对拍了几何深度学习图分支的完整骨架。下一步：拿起 PyG/DGL，去真实大图上把它们跑起来！